# CHD-VLM Evaluation — API Models

**Notebook 2 of 3** — Evaluate closed-source API models. **No GPU required.**

**Models**: GPT-4o, Claude-3.5-Sonnet, Gemini-1.5-Pro  
**Prompts**: ZSD, RCE, CoT, RAC  
**Dataset**: CHD-CXR (828 pediatric chest X-rays, 4 classes)  

**Estimated cost** (full run, all 3 models × 4 prompts × 828 samples):  
- GPT-4o: ~\$15–20  
- Claude-3.5-Sonnet: ~\$20–25  
- Gemini-1.5-Pro: ~\$5–10  
- **Total: ~\$40–55**

Run pilot first (`pilot.yaml`) to estimate costs before committing to full run.

After this notebook, merge results with HF results and run `03_analysis.ipynb`.

## 0. Setup

In [ ]:
import os

REPO_URL = "https://github.com/rushankgoyal/chd-eval.git"
REPO_DIR = "/content/chd-eval"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/new

# Install only API dependencies (skip torch/transformers for speed)
!pip install -q Pillow pandas numpy scikit-learn scipy matplotlib seaborn tqdm pyyaml python-dotenv openai anthropic google-generativeai
print("Installation complete.")

## 1. API Key Setup

In [ ]:
from google.colab import userdata
import os

# Set API keys from Colab Secrets (add them in the key icon on the left sidebar)
os.environ["OPENAI_API_KEY"]    = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GOOGLE_API_KEY"]    = userdata.get("GOOGLE_API_KEY")

# Quick connectivity check
import openai, anthropic, google.generativeai as genai

try:
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    models = client.models.list()
    print("OpenAI: OK")
except Exception as e:
    print(f"OpenAI: FAILED — {e}")

try:
    ac = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    print("Anthropic: OK")
except Exception as e:
    print(f"Anthropic: FAILED — {e}")

try:
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    print("Google AI: OK")
except Exception as e:
    print(f"Google AI: FAILED — {e}")

## 2. Dataset Loading

In [ ]:
import sys
sys.path.insert(0, "/content/chd-eval/new")

# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/chd-cxr-dataset"

# Option B: Upload directly
# from google.colab import files
# uploaded = files.upload()
# !unzip -q chd-cxr.zip -d /content/data
# DATA_ROOT = "/content/data/chd-cxr"

DATA_ROOT = "/content/data/chd-cxr"  # <-- EDIT THIS

from src.data.dataset import load_samples_from_directory, LABELS

samples = load_samples_from_directory(DATA_ROOT)
print(f"\nTotal samples: {len(samples)}")
for label in LABELS:
    print(f"  {label}: {sum(s.label == label for s in samples)}")

## 3. Configuration & Cost Estimate

In [ ]:
from src.evaluation.runner import ExperimentConfig

# Start with pilot to verify and estimate costs, then switch to api_full
EXPERIMENT = "configs/experiments/api_full.yaml"  # <-- EDIT THIS
# EXPERIMENT = "configs/experiments/pilot.yaml"  # smoke test first

config = ExperimentConfig.from_yaml(EXPERIMENT)
n_samples = config.max_samples or len(samples)
n_conditions = len(config.model_configs) * len(config.prompt_ids)
total_calls = n_conditions * n_samples

print(f"Models    : {[m.display_name for m in config.model_configs]}")
print(f"Prompts   : {config.prompt_ids}")
print(f"Conditions: {n_conditions}")
print(f"Samples   : {n_samples}")
print(f"Total API calls: {total_calls:,}")
print(f"Results dir: {config.results_dir}")

## 4. Evaluation

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

from src.evaluation.runner import EvaluationRunner

runner = EvaluationRunner(config, samples)
df = runner.run()

print(f"\nResults shape: {df.shape}")
df.head()

In [ ]:
# Cost summary
print(f"\nTotal API cost: ${runner.cost_tracker.total_cost_usd:.4f}")
print("\nCost breakdown:")
print(runner.cost_tracker.summary().to_string(index=False))

## 5. Merge with HF Results

In [ ]:
import pandas as pd
from pathlib import Path

# Upload the HF results CSV from notebook 01
# from google.colab import files
# uploaded = files.upload()  # upload results/hf_full/combined.csv
# hf_df = pd.read_csv(list(uploaded.keys())[0])

# OR load from drive / path:
HF_RESULTS_PATH = None  # e.g. "/content/drive/MyDrive/hf_combined.csv"

api_df = df

if HF_RESULTS_PATH and Path(HF_RESULTS_PATH).exists():
    hf_df = pd.read_csv(HF_RESULTS_PATH)
    combined = pd.concat([hf_df, api_df], ignore_index=True)
    print(f"HF results: {len(hf_df)} rows")
    print(f"API results: {len(api_df)} rows")
    print(f"Combined: {len(combined)} rows")
    print(f"Models: {combined['model_name'].unique().tolist()}")
    
    Path("results").mkdir(exist_ok=True)
    combined.to_csv("results/combined_all_models.csv", index=False)
    print("\nSaved: results/combined_all_models.csv")
else:
    print("HF results not provided — API-only combined file saved.")
    api_df.to_csv("results/combined_api_only.csv", index=False)
    combined = api_df

In [ ]:
# Download combined results
from google.colab import files

out_path = "results/combined_all_models.csv"
if not Path(out_path).exists():
    out_path = "results/combined_api_only.csv"

print(f"Downloading: {out_path}")
files.download(out_path)